In [1]:
from langgraph.graph import StateGraph
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict
from dotenv import load_dotenv
import os

c:\Users\hshinde3\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()

True

In [3]:
model = ChatGoogleGenerativeAI(model='gemini-2.5-flash')

In [ ]:
model.invoke("Distance between moon and earth?").content

AIMessage(content="The average distance between the Earth and the Moon is approximately **384,400 kilometers (238,900 miles)**.\n\nHowever, this distance isn't constant because the Moon's orbit around the Earth is elliptical, not perfectly circular. So, the distance varies:\n\n*   **Perigee (closest point):** About 363,104 km (225,623 miles)\n*   **Apogee (farthest point):** About 406,696 km (252,088 miles)\n\nIt takes light about 1.28 seconds to travel this average distance.", additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'safety_ratings': []}, id='run--019d8b56-9aad-74d2-8f36-20e30a4dacda-0', usage_metadata={'input_tokens': 7, 'output_tokens': 144, 'total_tokens': 773, 'input_token_details': {'cache_read': 0}})

In [5]:
class BlogState(TypedDict):
    title:str
    outline:str
    content:str

In [7]:
graph = StateGraph(BlogState)

In [9]:
def create_outline(state:BlogState):
    topic = state['title']
    prompt = f"Create a detailed outline for a blogpost on topic {topic}"
    outline = model.invoke(prompt).content
    state['outline']=outline
    return state  

def create_content(state:BlogState):
    topic = state['title']
    outline = state['outline']
    prompt = f"Create a blog post on title: {topic}. The outline for the blogpost is: {outline}"
    content = model.invoke(prompt).content
    state['outline']=content
    return state   

In [ ]:
def check_seo(state:BlogState):
    topic = state['title']
    content = state['content']
        
    prompt = f"You are a professional blog content analyzer. Analyze this blog on the topic: {topic} for SEO: {content}. Check for headers and keyword flow."
    response = model.invoke(prompt) 
    return {"seo_report": response}

def clarity_critic(state: BlogState):
    """Checks for fluff, repetition, and logical clarity."""
    topic = state['title']
    outline = state['outline']
    content = state['content']
    prompt = (
        f"You are a professional blog content copyeditor. Review this blog on topic {topic}: {content}\n"
        "Identify:\n"
        "1. Redundancy: Are there sentences that say the same thing twice?\n"
        "2. Jargon: Are there overly complex words that could be simpler?\n"
        "3. Flow: Does each point lead naturally to the next?\n"
        "Provide a 'Clarity Score' (1-10) and suggestions for cutting the fluff."
    )
    
    response = model.invoke(prompt)
    return {"clarity_report": response.content}

def engagement_critic(state: BlogState):
    """Assesses if the post is boring or actually provides value to a human."""

    topic = state['title']
    outline = state['outline']
    content = state['content']
    prompt = (
        f"You are a critical blog content reader. Analyze this blog post on the topic {topic}: content\n"
        "Rate 1-10 on: \n"
        "1. Hook: Does the first paragraph grab attention?\n"
        "2. Actionability: Can the reader actually do something with this info?\n"
        "3. CTA: Is the ending persuasive?\n"
        "Return a brief summary of what could be better."
    )
    # Simple LLM call
    response = model.invoke(prompt)
    return {"engagement_report": response.content}

In [ ]:
graph.add_node('create_outline', create_outline)
graph.add_node('create_content', create_content)

graph.add_node('check_seo', check_seo)